In [14]:
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_openai import ChatOpenAI

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5, max_tokens=256)

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are helpful assistant"),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

chat = prompt | llm
history = []


def ask(question):
    reply = chat.invoke({"history": history, "question": question})
    history.append(HumanMessage(question))
    history.append(AIMessage(reply.content))
    return reply.content


print(ask(question="what is the capital of france"))
print(ask(question="what is the population"))


print(history)

The capital of France is Paris.
As of my last update in October 2023, the population of Paris is approximately 2.1 million people within the city limits. However, if you consider the larger metropolitan area, the population is around 12 million. For the most current population figures, please check the latest statistics from reliable sources.
[HumanMessage(content='what is the capital of france', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is the population', additional_kwargs={}, response_metadata={}), AIMessage(content='As of my last update in October 2023, the population of Paris is approximately 2.1 million people within the city limits. However, if you consider the larger metropolitan area, the population is around 12 million. For the most current population figures, please check the latest statistics from reliable sour

In [ ]:
prompt = PromptTemplate.from_template(
    "Explain this {topic} in an Explain like I am 5 way (ELI5)"
)
parser = StrOutputParser()
chain = prompt | llm | parser
res = chain.invoke({"topic": "volcanoes"})
print(res)

Sure! Imagine the Earth is like a big ball made of different layers, kind of like an onion. Deep inside the Earth, it's really hot, and there's melted rock called magma. Sometimes, this magma wants to come out because it gets too crowded down there.

When the magma pushes its way up through cracks in the Earth’s surface, it can create a volcano. Think of a volcano like a big mountain that can spit out lava, ash, and gases when it erupts. 

When the volcano erupts, it’s like a big, exciting explosion! The lava flows down the sides and can make new land. After a while, the volcano might stop erupting and can become quiet, like a sleeping giant. 

So, volcanoes are nature’s way of letting out some of that hot, melted rock from deep inside the Earth!


In [ ]:
# Take a input and run these in parallel

# Run Summary
summary_prompt = PromptTemplate.from_template("Summarize in one sentence: {text}")
# Run Translation
translation_prompt = PromptTemplate.from_template("Translate to French: {text}")
# Run Sentiment
sentiment_prompt = PromptTemplate.from_template(
    "Is the sentiment of this text positive, negative or neutral? One word only: {text}"
)

chain = RunnableParallel(
    summary=summary_prompt | llm | StrOutputParser(),
    sentiment=sentiment_prompt | llm | StrOutputParser(),
    translation=translation_prompt | llm | StrOutputParser(),
)

chain.invoke({"text": "I love this product"})

{'summary': 'I am very fond of this product.',
 'sentiment': 'Positive',
 'translation': "J'adore ce produit."}

In [ ]:
template = "Tell me a {adjective} joke about {topic}"


def format_prompt(inputs):
    return template.format(adjective=inputs["adjective"], topic=inputs["topic"])


joke_chain = RunnableLambda(format_prompt) | llm | StrOutputParser()

res = joke_chain.invoke({"adjective": "silly", "topic": "robots"})

KeyError: 'topic'